# Ideas planner, second list, idea 5: re-run skipped photos on altered copies instead of estimating the alteration
No estimator. A photo that the standard 6 queries skip is run again on (a) a copy with the polarity flipped and (b) a copy sharpened by an unsharp mask (sigma 2 px, amount 1.5). The existing decision (bar 8, or two queries agreeing) decides each copy; if several copies answer, the one with the most inliers wins.
Photos: the 60 official photos of Test_Iteration_9, each as clean, polarity-flipped ('invert') and blurred 2 px ('blur'), the two alterations that cost 9-30 and 20-29 points in the red team (T9). First pass only (6 queries); the 27-query escalation is not added here.
Reading: points of the first pass alone against first pass + copies, wrong answers (error > 8 sigma, -20), and copies that answered a CLEAN skipped photo wrongly (the safety check).
Falsification (planned): dead if the copies recover less than half of the lost points on invert and blur, or give any wrong answer.

In [1]:
import os, sys, time, gc, functools
from pathlib import Path
import numpy as np
import pandas as pd
import torch

ROOT = Path.cwd()
while not (ROOT / "single_photo_pipeline.py").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT)); sys.path.insert(0, str(ROOT / "src")); os.chdir(ROOT)
T7 = ROOT / "tests/Test_Iteration_7/results"

DATA = os.environ.get("GATE_DATA", "official")          # official or nudt
POOL_PHOTOS = int(os.environ.get("POOL_PHOTOS", "500"))
TAG = os.environ.get("TAG", "official150")              # which rich dump (results/rich_dump/<TAG>) names the photos
N_LIMIT = int(os.environ.get("N_LIMIT", "0"))           # >0: only the first photos (smoke test)
print(DATA, TAG, "pool", POOL_PHOTOS, "limit", N_LIMIT)

official official150 pool 500 limit 0


In [2]:
import cv2
os.environ.setdefault("PHOTOS_FILE", str(ROOT / "tests/Test_Iteration_9/results/photos_official.csv"))

'/home/voidness/Storage/Synced Folders/InterIIT Tech 15/IdeaForge-GeoLoc-InterIIT15/tests/Test_Iteration_9/results/photos_official.csv'

In [3]:
if DATA == "nudt":
    from train.datasets import photo_sets
    photo_sets.load_nudt_photos = functools.partial(photo_sets.load_nudt_photos, split=os.environ.get("NUDT_SPLIT", "test"))
from train.checks.rich_dump import RichDumpPipeline
from train.checks import replay
from coarse_search.heading_queries import query_geometry
from image_processing.geometry import render_query
from image_processing.map_levels import level_for_height, level_gsd

def make_pipeline(headings, scales, confidence=0.22, coarse_threshold=0.21):
    pipeline = RichDumpPipeline(DATA, "edge_zncc", semantic_layer=True, semantic_decision=True, gamma=1.0, k_peaks=50, m_fine=10,
                                matcher="loftr_minima", min_inliers=8, ransac_pixels=None, matcher_confidence=confidence, satellite="new",
                                map_margin_m=None, loftr_coarse_threshold=coarse_threshold, dino_weight=None, headings=headings, scale_factors=scales)
    start = time.time()
    pipeline.setup(POOL_PHOTOS, 7)
    print("setup %.0f s" % (time.time() - start))
    return pipeline

STANDARD = [(-5.0, 1.0), (0.0, 1.0), (5.0, 1.0), (-5.0, 1.05), (0.0, 1.05), (5.0, 1.05)]
dump = None

In [4]:
pipeline = make_pipeline([-5.0, 0.0, 5.0], [1.0, 1.05])
from train.checks.perturb import perturbed_item
from train.checks.rich_dump import pack_photo, load_rich
from image_processing.thermal import load_thermal_frame
test_set = pipeline.test_set
indices = sorted(int(i) for i in pd.read_csv(os.environ["PHOTOS_FILE"]).photo)
if N_LIMIT:
    indices = indices[:N_LIMIT]
OUT_DIR = T7 / "perturbed_photos" / "idea5b"
SCRATCH = T7 / "rich_dump" / "idea5b_scratch"; SCRATCH.mkdir(parents=True, exist_ok=True)

def sharpened_item(item, out_dir):
    new = dict(item)
    os.makedirs(out_dir, exist_ok=True)
    path = os.path.abspath(os.path.join(out_dir, os.path.splitext(os.path.basename(str(item["name"])))[0] + "_sharp.png"))
    if not os.path.exists(path):
        frame = load_thermal_frame(item)
        f = frame.astype(np.float64)
        g = f + 1.5 * (f - cv2.GaussianBlur(f, (0, 0), 2.0))
        info = np.iinfo(frame.dtype) if np.issubdtype(frame.dtype, np.integer) else None
        if info is not None:
            g = np.clip(np.rint(g), info.min, info.max)
        assert cv2.imwrite(path, g.astype(frame.dtype)), path
    new["name"] = path
    return new

def first_pass(item, index):
    """Standard 6 queries of one photo, the decision on them (no escalation) and its error."""
    found = pipeline.run_queries(item, test_set.priors[index])
    arrays = pack_photo(item, test_set.truths[index], test_set.sigmas[index], found["level"], found["queries"], min_keep_confidence=0.15)
    path = SCRATCH / "photo.npz"; np.savez_compressed(path, **arrays)
    photo = load_rich(path, with_matches=False)
    return replay.decide_current(photo, escalate=False)

level 3: 0.85 m/pixel, picture 16378 x 16005, map coverage 100%
level 4: 1.20 m/pixel, picture 11581 x 11317, map coverage 100%
level 5: 1.70 m/pixel, picture 8189 x 8002, map coverage 100%
level 6: 2.40 m/pixel, picture 5790 x 5658, map coverage 100%
level 7: 3.39 m/pixel, picture 4094 x 4001, map coverage 100%


map store ready, took 111 s


/home/voidness/Storage/Synced Folders/InterIIT Tech 15/venv-geoloc/lib/python3.10/site-packages/kornia/feature/lightglue.py:44: FutureWarning: `torch.cuda.amp.custom_fwd(args...)` is deprecated. Please use `torch.amp.custom_fwd(args..., device_type='cuda')` instead.
  @torch.cuda.amp.custom_fwd(cast_inputs=torch.float32)


/home/voidness/Storage/Synced Folders/InterIIT Tech 15/venv-geoloc/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


setup 127 s


In [5]:
rows = []
start = time.time()
for kind in ["clean", "invert", "blur"]:
    for index in indices:
        base = dict(test_set.items[index]) if kind == "clean" else perturbed_item(test_set.items[index], kind, str(OUT_DIR / kind), load_thermal_frame)
        first = first_pass(base, index)
        row = dict(kind=kind, photo=index, height=float(test_set.items[index]["rel_height"]), stage=first["stage"], inliers=first["inliers"],
                   points_first=first["points"], wrong_first=int(first["answered"] and first["err_sigma"] > 8))
        copies = {}
        if not first["answered"]:
            copies["flip"] = first_pass(perturbed_item(base, "invert", str(OUT_DIR / ("copy_flip_" + kind)), load_thermal_frame), index)
            copies["sharp"] = first_pass(sharpened_item(base, str(OUT_DIR / ("copy_sharp_" + kind))), index)
        answered = {n: c for n, c in copies.items() if c["answered"]}
        if answered:
            name = max(answered, key=lambda n: answered[n]["inliers"])
            row.update(copy=name, copy_inliers=answered[name]["inliers"], points_copy=answered[name]["points"], wrong_copy=int(answered[name]["err_sigma"] > 8))
        else:
            row.update(copy="", copy_inliers=0.0, points_copy=0, wrong_copy=0)
        rows.append(row)
        torch.cuda.empty_cache(); gc.collect()
    print(kind, "done %.0f s" % (time.time() - start), flush=True)
C = pd.DataFrame(rows)
C["points_total"] = np.where(C.points_first > 0, C.points_first, C.points_copy)
C["wrong_total"] = C.wrong_first + C.wrong_copy
C.to_csv(T7 / "idea5b_altered_copies.csv", index=False)
summary = C.groupby("kind").agg(photos=("photo", "count"), first_answered=("stage", lambda s: int((s != "skip").sum())), score_first=("points_first", lambda p: round(p.sum() / len(p) * 6, 2)),
                                  score_with_copies=("points_total", lambda p: round(p.sum() / len(p) * 6, 2)), copies_answered=("copy", lambda c: int((c != "").sum())),
                                  wrong_first=("wrong_first", "sum"), wrong_total=("wrong_total", "sum")).reset_index()
summary.to_csv(T7 / "idea5b_altered_copies_summary.csv", index=False)
print(summary.to_string(index=False))

clean done 272 s


invert done 656 s


blur done 1141 s


  kind  photos  first_answered  score_first  score_with_copies  copies_answered  wrong_first  wrong_total
  blur      60              29         25.2               38.2               11            1            1
 clean      60              56         54.8               56.4                2            0            0
invert      60              42         39.2               53.8               15            0            0
